# EMO1 - series areales
***

***Autor:** Jesús Casado Rodríguez*<br>
***Fecha:** 27-03-2026*<br>

**Introducción:**<br>
En este _notebook_ se calculan las series meteorológicas areales de las cuencas de CAMELS-ES a partir de los datos meteorológicos de EMO1, los datos de entrada del sistema EFASv5. 

Las series meteorológicas generadas serán los datos dinámicos de entrada para el LSTM que replique el modelo hidrológico LISFLOOD, el usado en EFASv5.

**Por hacer:**<br>
* [x] Las series empiezan el 1 de enero de 1990 y terminan el 1 de enero del 2020. Deberían empezar el 1 de octubre de 2021 y terminar el 30 de septiembre de 2020.

In [ ]:
import geopandas as gpd

from ocab.config import Config
from ocab.basins.stats import read_data, read_pixarea, basin_statistics

# load configuration file
cfg = Config('../config_CAMELS.yml')

# load basins shapefile
basins = gpd.read_file(
    cfg.path_dataset / 'preprocessing' / 'basins' / 'output' / 'stations_basins_1min.geojson'
    ).set_index('ID')

# load pixel area to weigh the statistics
pixarea = read_pixarea(cfg.path_efas / 'maps' / 'pixarea_iberian_01min.nc')
pixarea = pixarea.rio.write_crs(cfg.crs)

# mload meteorological data
zarr_store = cfg.path_meteo / 'EMO1_1990-2022.zarr'
data = read_data(zarr_store, engine='zarr')
data = data.rio.write_crs(cfg.crs)
print(f"{data.nbytes / 1e9:.2f} GB")

# compute statistics
results = basin_statistics(
    data=data,
    basins=basins,
    statistic='mean',
    weight=pixarea,
    decimals=1,
    output=cfg.path_dataset / 'preprocessing' / 'catchstats' / 'meteo'
)